In [1]:
import numpy as np
import torch
import random
import os
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)
os.environ['PYTHONHASHSEED'] = str(0)

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import json
from latex_table_utils import format_table
from egci_bioacoustic_shifts import plot_EGCI
"""
region
    svm
    div
    data
        soundscape [complexity, entropy, empty]
        focal [complexity, entropy, empty]

"""

only_1_species_selected = False
#Inital Experiment

if not only_1_species_selected:
    with open("e1_results_more_examples.json", "r") as f:
        e1_results = json.load(f)
    with open("e1_results_SSW.json", "r") as f:
        ssw_results = json.load(f)

    e1_results["SSW"] = ssw_results["SSW"]

# Second Experiment with 1 species in soundscapes
else:
    with open("e1_results_True.json", "r") as f:
        e1_results = json.load(f)

In [3]:
from scipy.stats import mannwhitneyu
# Discovered when reloading an old checkpoint a single data point in focal_out SNE was none. Handle this case or others like it
def address_corruption(data):
    return [item for item in data if item is not None]


experiment_metrics = {}
for region in e1_results.keys():
    soundscape = pd.DataFrame(address_corruption(e1_results[region]["svm"]["soundscape_out"]))
    soundscape["GT"] = soundscape["gt"].apply(len)
    focal = pd.DataFrame(address_corruption(e1_results[region]["svm"]["focal_out"]))
    focal["GT"] = focal["gt"].apply(len)

    only_bird_soundscapes = soundscape[soundscape["GT"] > 0]

    if not only_1_species_selected:
        features = ['entropy', 'complexity', 'acoustic_complexity', 'acoustic_diversity',  'bioacoustic_index', 'GT']
        print(f"ALL BIRDS {region}")
        no_bird_soundscapes = soundscape[soundscape["GT"] == 0]
    else:
        features = ['entropy', 'complexity', 'acoustic_complexity', 'acoustic_diversity',  'bioacoustic_index']
        print(f"JUST 1 BIRD {region}")
        # Ignore the results from no bird, below is done just for the algorithm to work correctly
        no_bird_soundscapes = soundscape
    

    experiment_metrics[region] = {
            "soundscape_mwu": mannwhitneyu(focal[features], soundscape[features], method="asymptotic"),
            "only_bird_soundscape_mwu": mannwhitneyu(focal[features], only_bird_soundscapes[features], method="asymptotic"),
            "no_bird_soundscape_mwu": mannwhitneyu(focal[features], no_bird_soundscapes[features], method="asymptotic"),
        }   
experiment_metrics

ALL BIRDS HSN
ALL BIRDS PER
ALL BIRDS UHH
ALL BIRDS SNE
ALL BIRDS POW
ALL BIRDS NES
ALL BIRDS SSW


{'HSN': {'soundscape_mwu': MannwhitneyuResult(statistic=array([2327710., 3569287., 3838392.,  877855., 2707892., 2971000.]), pvalue=array([2.87049065e-019, 0.00000000e+000, 0.00000000e+000, 2.45188695e-207,
         1.05364204e-083, 8.51734853e-238])),
  'only_bird_soundscape_mwu': MannwhitneyuResult(statistic=array([1041556., 1582110., 1674489.,  399924., 1207995.,  755000.]), pvalue=array([5.53674102e-013, 8.46177720e-243, 0.00000000e+000, 1.87912584e-124,
         2.00987714e-052, 4.61585692e-072])),
  'no_bird_soundscape_mwu': MannwhitneyuResult(statistic=array([1286154., 1987177., 2163903.,  477931., 1499897., 2216000.]), pvalue=array([1.04449165e-013, 9.79468330e-295, 0.00000000e+000, 2.15147795e-152,
         3.96708626e-060, 0.00000000e+000]))},
 'PER': {'soundscape_mwu': MannwhitneyuResult(statistic=array([1319566., 1416031., 3540090.,  499501., 1607981.,  797000.]), pvalue=array([1.76429427e-077, 1.48503690e-057, 0.00000000e+000, 0.00000000e+000,
         7.00768205e-027, 6.4

In [4]:
def get_mwu_table(region, type):
        df = pd.DataFrame(data={f"{region} stat": experiment_metrics[region][type].statistic, "pvalue": experiment_metrics[region][type].pvalue}).T
        df.columns = features
        return df

mwu_all = []
mwu_bird = []
for region in experiment_metrics.keys():
    mwu_all.append(get_mwu_table(region, "soundscape_mwu"))
    mwu_bird.append(get_mwu_table(region, "only_bird_soundscape_mwu"))



og_table = pd.concat(mwu_all).to_latex()
print("ready_use_format")
print(og_table)
format_table(og_table)

ready_use_format
\begin{tabular}{lrrrrrr}
\toprule
 & entropy & complexity & acoustic_complexity & acoustic_diversity & bioacoustic_index & GT \\
\midrule
HSN stat & 2327710.000000 & 3569287.000000 & 3838392.000000 & 877855.000000 & 2707892.000000 & 2971000.000000 \\
pvalue & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 \\
PER stat & 1319566.000000 & 1416031.000000 & 3540090.000000 & 499501.000000 & 1607981.000000 & 797000.000000 \\
pvalue & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 \\
UHH stat & 2827877.000000 & 3444658.000000 & 3065323.000000 & 1397570.000000 & 2449056.000000 & 2044000.000000 \\
pvalue & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.126299 \\
SNE stat & 2959677.000000 & 3470035.000000 & 3866855.000000 & 383939.000000 & 3345341.000000 & 1986000.000000 \\
pvalue & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.639375 \\
POW stat & 1472013.000000 & 2884247.000000 & 2978828.000000 & 454947.000000 & 1544542.000

'\\begin{adjustbox}{width=\\columnwidth,center}\n \\begin{tabular}{l|llllll}\n  & {entropy} & {complexity} & {acoustic_complexity} & {acoustic_diversity} & {bioacoustic_index} & {GT} \\\\\n \\hline\n HSN stat & 2327710.000000 & 3569287.000000 & 3838392.000000 & 877855.000000 & 2707892.000000 & 2971000.000000 \\\\\n pvalue & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} \\\\\n \\hline\n PER stat & 1319566.000000 & 1416031.000000 & 3540090.000000 & 499501.000000 & 1607981.000000 & 797000.000000 \\\\\n pvalue & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} \\\\\n \\hline\n UHH stat & 2827877.000000 & 3444658.000000 & 3065323.000000 & 1397570.000000 & 2449056.000000 & 2044000.000000 \\\\\n pvalue & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & 0.126299 \\\\\n \\hline\n SNE stat & 2959677.000000 & 3470035.000000 

In [5]:
og_table = pd.concat(mwu_bird).to_latex()
print("ready_use_format")
print(og_table)
format_table(og_table)

ready_use_format
\begin{tabular}{lrrrrrr}
\toprule
 & entropy & complexity & acoustic_complexity & acoustic_diversity & bioacoustic_index & GT \\
\midrule
HSN stat & 1041556.000000 & 1582110.000000 & 1674489.000000 & 399924.000000 & 1207995.000000 & 755000.000000 \\
pvalue & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 \\
PER stat & 1192028.000000 & 1208638.000000 & 3172313.000000 & 455636.000000 & 1431847.000000 & 375000.000000 \\
pvalue & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 \\
UHH stat & 1958085.000000 & 2381244.000000 & 2142596.000000 & 964389.000000 & 1734237.000000 & 884000.000000 \\
pvalue & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 \\
SNE stat & 2009289.000000 & 2402011.000000 & 2663154.000000 & 223668.000000 & 2310652.000000 & 752000.000000 \\
pvalue & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 \\
POW stat & 1421795.000000 & 2795384.000000 & 2875229.000000 & 448901.000000 & 1496715.000000 

'\\begin{adjustbox}{width=\\columnwidth,center}\n \\begin{tabular}{l|llllll}\n  & {entropy} & {complexity} & {acoustic_complexity} & {acoustic_diversity} & {bioacoustic_index} & {GT} \\\\\n \\hline\n HSN stat & 1041556.000000 & 1582110.000000 & 1674489.000000 & 399924.000000 & 1207995.000000 & 755000.000000 \\\\\n pvalue & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} \\\\\n \\hline\n PER stat & 1192028.000000 & 1208638.000000 & 3172313.000000 & 455636.000000 & 1431847.000000 & 375000.000000 \\\\\n pvalue & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} \\\\\n \\hline\n UHH stat & 1958085.000000 & 2381244.000000 & 2142596.000000 & 964389.000000 & 1734237.000000 & 884000.000000 \\\\\n pvalue & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} & \\maxf{0.000000} \\\\\n \\hline\n SNE stat & 2009289.000000 & 2402011.00